In [285]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Версия Python:", sys.version)
print("Путь к интерпретатору:", sys.executable)
print('numpy :', np.__version__)
print('pandas:', pd.__version__)
print('seaborn:', sns.__version__)

Версия Python: 3.12.16 | packaged by conda-forge | (main, Oct 26 2026, 11:29:01) [MSC v.1944 64 bit (AMD64)]
Путь к интерпретатору: C:\Users\6anna\miniconda3\envs\ds-lab1\python.exe
numpy : 2.5.4
pandas: 3.1.0
seaborn: 0.13.2


Проверим расположение clients.csv и transactions.csv

In [286]:
# Проверка наличия данных
from pathlib import Path

DATA_DIR = Path("data/raw")

CLIENTS_PATH = DATA_DIR / "clients.csv"
TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"

print("Папка:", DATA_DIR.absolute())
print("clients.csv:", CLIENTS_PATH.is_file())
print("transactions.csv:", TRANSACTIONS_PATH.is_file())

Папка: C:\Users\6anna\PycharmProjects\Suhodolova_Anna_Lab1\data\raw
clients.csv: True
transactions.csv: True


## 1. Аудит качества данных

## 1.1 Наивная загрузка
Читаем данные без разделителей и типов

In [287]:
if CLIENTS_PATH.is_file() and TRANSACTIONS_PATH.is_file():
    df_clients_naive = pd.read_csv(CLIENTS_PATH, encoding='utf-8')
    df_transactions_naive = pd.read_csv(TRANSACTIONS_PATH, encoding='cp1251')
    print("\nclients.shape:", df_clients_naive.shape)
    print("transactions.shape:", df_transactions_naive.shape)
    
    print(df_clients_naive.dtypes.value_counts())
    print(df_transactions_naive.dtypes.value_counts())
    
    print(df_clients_naive.head(3))
    print(df_transactions_naive.head(5))
else:
    print("Files not found")
    


clients.shape: (5702, 16)
transactions.shape: (629930, 1)
str        13
float64     2
int64       1
Name: count, dtype: int64
str    1
Name: count, dtype: int64
  client_id  birth_date gender             city      segment loyalty_tier  \
0   C004056  1964-02-02      Ж     Екатеринбург  Премиальный       Золото   
1   C003283  1984-09-15      М  Нижний Новгород     Массовый      Серебро   
2   C000528  1979-01-20      М      Н. Новгород   Зарплатный      Базовый   

   card_type registration_date card_open_date            income_source  \
0  Дебетовая        25.05.2015     08.11.2019   Зарплата в нашем банке   
1  Дебетовая        08.05.2015     14.12.2016  Зарплата в другом банке   
2  Дебетовая        01.09.2016     26.06.2021   Зарплата в нашем банке   

   monthly_income has_mobile_app app_last_login  satisfaction_score  \
0        240200.0            Нет            NaN                 NaN   
1         84100.0            Нет            NaN                 NaN   
2        103300.0  

Вывод: 
- clients.csv весь прочитан в таблицу (5702, 16)
- transactions.csv прочитан некорректно: получен один столбец вместо отдельных полей, видно, что разделителем является ';'

In [288]:
# Типы всех столбцов
print("\n Типы данных всех столбцов:")
print(df_clients_naive.dtypes)


 Типы данных всех столбцов:
client_id                 str
birth_date                str
gender                    str
city                      str
segment                   str
loyalty_tier              str
card_type                 str
registration_date         str
card_open_date            str
income_source             str
monthly_income        float64
has_mobile_app            str
app_last_login            str
satisfaction_score    float64
branch_code             int64
marketing_opt_in          str
dtype: object


Вывод: типы требуют уточнения и не совсем корректны:
 - даты(birth_date, registration_date, card_open_date, app_last_login) представлены str, а не спец типом даты
  - satisfaction_score содержит целые оценки, а не дробные

In [289]:
# Количество пропусков (NaN) в каждом столбце
print("\n Пропуски (NaN) в каждом столбце:")
print(df_clients_naive.isnull().sum())


 Пропуски (NaN) в каждом столбце:
client_id                0
birth_date               0
gender                   0
city                     0
segment                  0
loyalty_tier             0
card_type                0
registration_date        0
card_open_date           0
income_source            0
monthly_income         558
has_mobile_app           0
app_last_login        2140
satisfaction_score    3416
branch_code              0
marketing_opt_in         0
dtype: int64


## 1.2 Правильная загрузка
Зададим типы полей и даты разберём отдельно форматам источника


In [290]:
df_clients = pd.read_csv(
    CLIENTS_PATH,
    encoding='utf-8',
    dtype={
        'client_id': 'str',
        'city': 'str',
        'gender': 'category',
        'segment': 'category',
        'loyalty_tier': 'category',
        'card_type': 'category',
        'income_source': 'category',
        'has_mobile_app': 'category',
        'marketing_opt_in': 'category',
        'branch_code': 'int64',
        'satisfaction_score': 'Int64',    #Это порядковая шкала, мне удобно Int так как порядок тогда уже задан 
        'monthly_income': 'float64',
    }
)
df_clients['birth_date'] = pd.to_datetime(
    df_clients['birth_date'], format='%Y-%m-%d', errors='coerce'
)
df_clients['registration_date'] = pd.to_datetime(
    df_clients['registration_date'], format='%d.%m.%Y', errors='coerce'
)
df_clients['card_open_date'] = pd.to_datetime(
    df_clients['card_open_date'], format='%d.%m.%Y', errors='coerce'
)
df_clients['app_last_login'] = pd.to_datetime(
    df_clients['app_last_login'], format='%Y-%m', errors='coerce'
)

df_transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    encoding='cp1251',
    sep=';',
    decimal=',',
    parse_dates=['txn_datetime'],
    dtype={
        'txn_id': 'str',
        'client_id': 'str',
        'currency': 'category',
        'category': 'category',
        'mcc_code': 'str',
        'channel': 'category',
        'status': 'category',
        'amount': 'float64',
    }
)

print("clients.shape:", df_clients.shape)
print("transactions.shape:", df_transactions.shape)


clients.shape: (5702, 16)
transactions.shape: (629930, 9)


## 1.3 Категории и обозначения отсутсвующих сведений
## Аудит и нормализация категориальные данные

In [291]:
# Список столбцов для анализа
cat_cols = [
    'gender', 'city', 'segment', 'loyalty_tier', 'card_type',
    'income_source', 'has_mobile_app', 'satisfaction_score', 'marketing_opt_in'
]

# Частота категорий в каждом столбце
for col in cat_cols:
    print(f"\n Столбец: {col}")
    print(df_clients[col].value_counts(dropna=False))  # dropna=False — показывает NaN тоже


 Столбец: gender
gender
Ж    2903
М    2799
Name: count, dtype: int64

 Столбец: city
city
Москва                1108
Екатеринбург           666
Санкт-Петербург        622
Новосибирск            557
Нижний Новгород        550
Казань                 534
г. Москва              199
 москва                195
МОСКВА                 177
Н. Новгород            166
казань                 161
г. Новосибирск         147
ЕКАТЕРИНБУРГ           134
г.Екатеринбург         134
СПб                    131
г. Санкт-Петербург     114
Санкт Петербург        107
Name: count, dtype: int64

 Столбец: segment
segment
Массовый       2893
Зарплатный     1687
Премиальный     884
Private         238
Name: count, dtype: int64

 Столбец: loyalty_tier
loyalty_tier
Базовый    2890
Серебро    1591
Золото      920
Платина     301
Name: count, dtype: int64

 Столбец: card_type
card_type
Дебетовая      3955
Кредитная      1275
Премиальная     472
Name: count, dtype: int64

 Столбец: income_source
income_source
Зарплат

Выводы: 
- Есть разные написания одних и тех же городов
- Нет мобильного приложения(has_mobile_app) у 2140 и у 2140 клиентов NaT в app_last_login, надо проверить одни и те же ли это клиенты
- 1549 клиентов не указали источник зарплаты. Нужно посмотреть опять же связи с другими признаками
- 648 клиента отказались от участия маркетинговой коммуникации, поэтому они не будут включены ни в одну из групп, так как они не интересны
- 3416 пропуска в satisfaction_score нельзя ничем заменить. Далее проверим есть ли связи с другими признаками

In [292]:
# Приведём все города к одному написанию
import re

def normalize_city(city):
    """Brings the city to its standard appearance"""
    if pd.isna(city):
        return city
    
    city = city.strip()  # убираем пробелы
    # Убираем "г. " или "город " в начале
    city = re.sub(r'^(г\.|город)\s*', '', city, flags=re.IGNORECASE)
    city = city.title() # Один регистр
    
    abbreviations = {
        'Спб': 'Санкт-Петербург',
        'Санкт Петербург': 'Санкт-Петербург',
        'Н.Новгород': 'Нижний Новгород',
        'Н. Новгород': 'Нижний Новгород',
        'Ростов-На-Дону': 'Ростов-на-Дону',
    }
    
    return abbreviations.get(city, city)

df_clients['city'] = df_clients['city'].apply(normalize_city)

print(df_clients['city'].value_counts(dropna=False))

city
Москва             1679
Санкт-Петербург     974
Екатеринбург        934
Нижний Новгород     716
Новосибирск         704
Казань              695
Name: count, dtype: int64


Вывод: разные написания одного и того же города привели к одному названию.

In [293]:
# Проверяет соответствие отсутствия даты последнего входа и отсутствие мобильного приложения 
mask_anomaly = (
    (df_clients['has_mobile_app'] == 'Нет') & (df_clients['app_last_login'].notna())  
    |
    (df_clients['has_mobile_app'] == 'Да') & (df_clients['app_last_login'].isna())    
)

df_anomalies = df_clients[mask_anomaly].copy()

print(f"Найдено аномальных записей: {len(df_anomalies)}")
print(f"Из {len(df_clients)} всего ({len(df_anomalies)/len(df_clients)*100:.2f}%)")

# Смотрим на аномалии
display(df_anomalies[['client_id', 'has_mobile_app', 'app_last_login']].head(20))

Найдено аномальных записей: 0
Из 5702 всего (0.00%)


,client_id,has_mobile_app,app_last_login


Вывод: даты последнего входа не указаны только у людей, у которых нет приложения. Данный пропуск ничем не заменяем

## 1.4 Повторные записи
Разбираем столбец client_id

In [294]:
# Проверим уникальность клиентов
n_clients_raw = len(df_clients)
n_unique_clients_raw = df_clients['client_id'].nunique()

print('Всего строк:', n_clients_raw)
print('Уникальных client_id:', n_unique_clients_raw)
print('Пропусков client_id:', df_clients['client_id'].isna().sum())
print('Строк с повторяющимся client_id:',
      df_clients['client_id'].duplicated(keep=False).sum())

fully_duplicated_rows = df_clients.duplicated().sum()
print('Лишних идентичных записей:', fully_duplicated_rows)

Всего строк: 5702
Уникальных client_id: 5647
Пропусков client_id: 0
Строк с повторяющимся client_id: 110
Лишних идентичных записей: 40


Вывод: обнаружено 40 полный дубликатов, я считаю, что нужно удалить дубликаты, так как очень маловероятно, что человек открыл две абсолютно одинаковые карты в один день(сейчас это никак нельзя проверить точно).
 Остальные частичные дубликаты с одинаковым client_id нужно проверить: какие признаки отличаются(т.е. возможно ли что это 2 или более карт у одного клиента)

In [295]:
# Удалим полные дубликаты
n_before = len(df_clients)
df_clients = df_clients.drop_duplicates().copy()
n_full_removed = n_before - len(df_clients)

print('Удалено полных дубликатов:', n_full_removed)
print('Осталось строк:', len(df_clients))

Удалено полных дубликатов: 40
Осталось строк: 5662


In [296]:
# Проверяем все неидентичные записи клиентов с повторяющимся ID
conflict_mask = df_clients['client_id'].duplicated(keep=False)

clients_conflicts = (
    df_clients.loc[conflict_mask]
    .sort_values('client_id')
    .copy()
)

print('Конфликтующих строк:', len(clients_conflicts))
print('Клиентов с конфликтом:',
      clients_conflicts['client_id'].nunique())

# Количество разных значений каждого поля внутри client_id
different = (
    clients_conflicts
    .groupby('client_id', observed=True)
    .nunique(dropna=False)
    .gt(1)
)

# Число клиентов с расхождениями по каждому полю
conflict_counts = different.sum()

display(
    conflict_counts[conflict_counts > 0]
    .to_frame('n_clients_with_conflict')
)

conflict_cols = different.columns[different.any()].tolist()
display(clients_conflicts[['client_id'] + conflict_cols])

Конфликтующих строк: 30
Клиентов с конфликтом: 15


,n_clients_with_conflict
city,15
segment,13


,client_id,city,segment
5024,C000828,Санкт-Петербург,Зарплатный
5256,C000828,Москва,Премиальный
3300,C001874,Москва,Массовый
2741,C001874,Нижний Новгород,Private
4615,C002496,Москва,Зарплатный
2082,C002496,Новосибирск,Массовый
2067,C002506,Нижний Новгород,Массовый
3054,C002506,Новосибирск,Зарплатный
5104,C002573,Екатеринбург,Массовый
3271,C002573,Москва,Массовый


Вывод: всего 30 неидентичных строк с одинаковым client_id, отличия: 13 по городам и 15 по сегментам, так как остальные поля, включая тип карты и дату открытия, совпадают значит нет повода считать эти строки как отдельные строки.
Так как нет вариантов определить какая запись актуальна, то эти 0.27% от 5647 уникальных клиентов исключим из основной выборки.

In [297]:
# сохраняем id конфликтующих клиентов для аудита
conflict_ids = clients_conflicts['client_id'].unique()

# Удаляем всех конфликтующих клиентов
df_clients = df_clients.loc[
    ~df_clients['client_id'].isin(conflict_ids)
].copy()

assert df_clients['client_id'].notna().all()
assert df_clients['client_id'].is_unique

print('Осталось строк:', len(df_clients))
print('Уникальных client_id:', df_clients['client_id'].nunique())
print('Исключено уникальных клиентов:', len(conflict_ids))


Осталось строк: 5632
Уникальных client_id: 5632
Исключено уникальных клиентов: 15


Вывод: в таблице осталось 5632 строчки с уникальными client_id 

## 1.5 Возраст и согласованность дат
Справочник представлен на момент 30.06.2025

In [298]:
REFERENCE_DATE = pd.Timestamp('2025-06-30')

# Проверим типы
date_cols = [
    'birth_date',
    'registration_date',
    'card_open_date',
    'app_last_login',
]

print(df_clients[date_cols].dtypes)

birth_date           datetime64[us]
registration_date    datetime64[us]
card_open_date       datetime64[us]
app_last_login       datetime64[us]
dtype: object


In [299]:
def full_years(birth_date, event_date):
    """
    Calculates the age in full years between the date of birth and the event
    """
    # Проверяем наступил ли день рождения в году события
    birthday_not_reached = (
        (event_date.dt.month < birth_date.dt.month)
        | ((event_date.dt.month == birth_date.dt.month)
           & (event_date.dt.day < birth_date.dt.day))
    )
    
    years = (
        event_date.dt.year - birth_date.dt.year
        - birthday_not_reached.astype('int64')
    )
    
    valid = birth_date.notna() & event_date.notna()
    return years.where(valid).astype('Int64')

# Создаём Series с контрольной датой для каждого клиента
reference_dates = pd.Series(REFERENCE_DATE, index=df_clients.index)

df_clients['age'] = full_years(df_clients['birth_date'], reference_dates)
df_clients['age_at_registration'] = full_years(
    df_clients['birth_date'], df_clients['registration_date'],
)
df_clients['age_at_card_open'] = full_years(
    df_clients['birth_date'], df_clients['card_open_date'],
)

In [300]:
display(df_clients['age'].describe())

age_cols = [
    'client_id', 'birth_date', 'age',
    'registration_date', 'age_at_registration',
    'card_open_date', 'age_at_card_open', 'card_type',
]

# Самые молодые
display(df_clients.sort_values('age')[age_cols].head(10))
# Самые пожилые
display(df_clients.sort_values('age', ascending=False)[age_cols].head(10))


count       5632.0
mean     52.812855
std      13.945073
min           23.0
25%           42.0
50%           51.0
75%           62.0
max           88.0
Name: age, dtype: Float64

,client_id,birth_date,age,registration_date,age_at_registration,card_open_date,age_at_card_open,card_type
2419,C004500,2002-06-22,23,2015-05-09,12,2016-09-15,14,Премиальная
34,C004863,2001-08-09,23,2013-03-30,11,2017-09-17,16,Дебетовая
685,C005219,2000-09-29,24,2016-03-04,15,2017-01-25,16,Кредитная
112,C003214,2001-06-23,24,2014-10-08,13,2017-07-07,16,Дебетовая
3068,C000054,2001-04-26,24,2017-09-22,16,2017-11-21,16,Дебетовая
5640,C004392,2000-11-06,24,2014-10-25,13,2019-07-14,18,Кредитная
1077,C003300,2000-08-23,24,2020-04-11,19,2024-08-27,24,Кредитная
728,C003513,2001-06-10,24,2017-02-14,15,2018-10-27,17,Дебетовая
5383,C001403,2000-01-08,25,2022-03-22,22,2022-05-01,22,Дебетовая
1927,C003886,1999-11-07,25,2015-06-23,15,2021-06-21,21,Кредитная


,client_id,birth_date,age,registration_date,age_at_registration,card_open_date,age_at_card_open,card_type
5682,C002903,1937-03-12,88,2017-04-26,80,2019-01-24,81,Премиальная
4994,C000382,1936-11-01,88,2018-11-12,82,2022-01-11,85,Премиальная
18,C003930,1937-01-08,88,2016-10-03,79,2022-02-23,85,Премиальная
501,C000990,1936-11-09,88,2018-12-19,82,2019-02-05,82,Дебетовая
765,C000222,1936-11-27,88,2014-11-09,77,2016-07-04,79,Дебетовая
834,C001917,1937-05-30,88,2015-04-24,77,2019-10-24,82,Кредитная
5529,C001014,1936-09-14,88,2018-02-09,81,2020-11-25,84,Дебетовая
489,C004997,1937-05-04,88,2018-08-10,81,2022-08-28,85,Премиальная
860,C003582,1936-10-08,88,2021-07-05,84,2025-01-01,88,Дебетовая
929,C005401,1936-11-19,88,2018-01-13,81,2022-06-19,85,Дебетовая


Вывод: возраст клиентов составляет от 23 до 88 лет. Эти возраста на данный момент полностью подходят для любых операций. Но нужно ещё проверить их возраст на момент регистрации и открытия карты

In [301]:
# Зафиксируем возрастные рамки для различных 
age_flags = pd.DataFrame({
    'birth_in_future': df_clients['birth_date'] > REFERENCE_DATE,
    'age_over_110': df_clients['age'] > 110,
    'age_under_14': df_clients['age'] < 14,
    'card_open_under_14': df_clients['age_at_card_open'] < 14,
    'credit_card_under_18': (
        df_clients['card_type'].eq('Кредитная')
        & df_clients['age_at_card_open'].lt(18)
    ),
}).fillna(False)

display(age_flags.sum().to_frame('n_rows'))
display(df_clients.loc[age_flags.any(axis=1), age_cols])

,n_rows
birth_in_future,0
age_over_110,0
age_under_14,0
card_open_under_14,0
credit_card_under_18,3


,client_id,birth_date,age,registration_date,age_at_registration,card_open_date,age_at_card_open,card_type
685,C005219,2000-09-29,24,2016-03-04,15,2017-01-25,16,Кредитная
4285,C003590,2000-05-11,25,2016-05-18,16,2017-12-31,17,Кредитная
5165,C004243,1999-12-31,25,2014-05-27,14,2016-01-02,16,Кредитная


Вывод: у трёх клиентов на данный момент тип карты указан как "кредитная", а возраст при открытии этой карты составляет 16-17 лет. Это может свидетельствовать либо о несогласованности атрибутов, либо о правонарушении, либо об эмансипации

In [302]:
date_flags = pd.DataFrame({
    'registration_before_birth': (
        df_clients['registration_date'] < df_clients['birth_date']
    ),
    'card_before_birth': (
        df_clients['card_open_date'] < df_clients['birth_date']
    ),
    'registration_in_future': (
        df_clients['registration_date'] > REFERENCE_DATE
    ),
    'card_open_in_future': (
        df_clients['card_open_date'] > REFERENCE_DATE
    ),
    'card_before_registration': ( # Карта открыта ДО регистрации 
        df_clients['card_open_date'] < df_clients['registration_date']
    ),
    'login_after_snapshot_month': ( #  Последний вход в приложение был ПОСЛЕ контрольного месяца
        df_clients['app_last_login'].dt.to_period('M')
        > REFERENCE_DATE.to_period('M')
    ),
    'login_before_registration_month': (# Последний вход в приложение был ДО месяца регистрации
        df_clients['app_last_login'].dt.to_period('M')
        < df_clients['registration_date'].dt.to_period('M')
    ),
}).fillna(False)

display(date_flags.sum().to_frame('n_rows'))

display(df_clients.loc[
    date_flags.any(axis=1),
    ['client_id', 'birth_date', 'registration_date',
     'card_open_date', 'has_mobile_app', 'app_last_login']
])

,n_rows
registration_before_birth,0
card_before_birth,0
registration_in_future,0
card_open_in_future,0
card_before_registration,0
login_after_snapshot_month,0
login_before_registration_month,15


,client_id,birth_date,registration_date,card_open_date,has_mobile_app,app_last_login
227,C000155,1957-07-11,2025-01-15,2025-03-07,Да,2024-07-01
761,C001729,1980-03-30,2024-05-12,2025-02-17,Да,2024-01-01
1066,C005485,1963-11-20,2024-06-10,2025-01-27,Да,2024-04-01
1128,C000150,1945-08-17,2024-11-27,2024-12-29,Да,2024-03-01
1416,C001197,1992-03-23,2024-05-01,2024-09-23,Да,2024-01-01
1871,C001654,1975-10-31,2025-02-06,2025-04-27,Да,2024-01-01
2063,C003665,1937-09-22,2024-11-14,2025-03-27,Да,2024-08-01
2413,C005988,1949-11-15,2024-10-07,2024-12-02,Да,2024-03-01
2420,C003597,1979-05-15,2024-06-22,2024-11-09,Да,2024-05-01
2704,C003577,1981-08-16,2025-02-02,2025-02-14,Да,2024-09-01


Вывод: у 15 клиентов месяц последнего входа предшествует месяцу регистрации. Ничего не заменяем просто запоминаем, что эта дата не всегда может показывать точно активность

## 1.6 Проверка ежемесячного дохода

In [303]:
display(df_clients['monthly_income'].describe())

print(f"\nПропусков (NaN): {df_clients['monthly_income'].isna().sum()}")
print(f"Доля пропусков: {df_clients['monthly_income'].isna().mean() * 100:.2f}%")

# Отрицательные значения и нули
print(f"\nОтрицательных значений: {(df_clients['monthly_income'] < 0).sum()}")
print(f"Нулевых значений: {(df_clients['monthly_income'] == 0).sum()}")

# Низкие положительные доходы: проверить
low_positive = (
    df_clients['monthly_income'].gt(0)
    & df_clients['monthly_income'].lt(10_000)
)

print(f"Положительных доходов меньше 10 000 ₽: {low_positive.sum()}")

# Крупные доходы: проверить
print(f"Больше 1 000 000 ₽: {(df_clients['monthly_income'] > 1_000_000).sum()}")
print(f"Больше 10 000 000 ₽: {(df_clients['monthly_income'] > 10_000_000).sum()}")

# Квантили (показывают распределение)
print("\n=== Квантили ===")
display(df_clients['monthly_income'].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

# Топ-10 самых высоких доходов (проверка на аномалии)
print("\n=== Топ-10 максимальных значений ===")
display(
    df_clients.nlargest(10, 'monthly_income')[
        ['client_id', 'monthly_income', 'segment', 'loyalty_tier', 'city']
    ]
)

# Топ-10 самых низких (не нулевых) доходов
print("\n=== Топ-10 минимальных значений (не нулевых) ===")
non_zero = df_clients[df_clients['monthly_income'] > 0]
display(
    non_zero.nsmallest(10, 'monthly_income')[
        ['client_id', 'monthly_income', 'segment', 'loyalty_tier', 'city']
    ]
)

count    5.080000e+03
mean     9.407863e+04
std      8.362391e+04
min     -9.990000e+02
25%      5.120000e+04
50%      7.200000e+04
75%      1.069000e+05
max      1.120600e+06
Name: monthly_income, dtype: float64


Пропусков (NaN): 552
Доля пропусков: 9.80%

Отрицательных значений: 60
Нулевых значений: 0
Положительных доходов меньше 10 000 ₽: 175
Больше 1 000 000 ₽: 2
Больше 10 000 000 ₽: 0

=== Квантили ===


0.01      -999.0
0.05     22400.0
0.25     51200.0
0.50     72000.0
0.75    106900.0
0.95    241640.0
0.99    445268.0
Name: monthly_income, dtype: float64


=== Топ-10 максимальных значений ===


,client_id,monthly_income,segment,loyalty_tier,city
1216,C002788,1120600.0,Private,Платина,Нижний Новгород
447,C001404,1001800.0,Private,Серебро,Казань
1090,C005456,904300.0,Private,Золото,Санкт-Петербург
1624,C003591,808700.0,Private,Серебро,Москва
4429,C001574,776700.0,Private,Золото,Санкт-Петербург
2989,C000704,754500.0,Private,Платина,Москва
5372,C003965,749800.0,Private,Платина,Москва
2125,C002183,730500.0,Private,Золото,Новосибирск
5004,C000316,725800.0,Private,Платина,Москва
1520,C003194,722900.0,Private,Золото,Новосибирск



=== Топ-10 минимальных значений (не нулевых) ===


,client_id,monthly_income,segment,loyalty_tier,city
793,C000575,23.9,Массовый,Базовый,Нижний Новгород
3347,C003630,26.5,Массовый,Базовый,Екатеринбург
2966,C005110,29.6,Массовый,Базовый,Екатеринбург
1354,C003525,29.8,Массовый,Базовый,Санкт-Петербург
1870,C000080,29.8,Массовый,Базовый,Москва
2183,C002271,31.3,Массовый,Базовый,Санкт-Петербург
4698,C002538,32.2,Массовый,Серебро,Санкт-Петербург
1664,C000531,33.1,Массовый,Базовый,Санкт-Петербург
2223,C005069,33.6,Массовый,Базовый,Екатеринбург
4067,C004964,33.7,Зарплатный,Серебро,Нижний Новгород


In [304]:
negative_income = df_clients['monthly_income'].lt(0)

print('Частоты отрицательных значений:')
display(
    df_clients.loc[negative_income, 'monthly_income']
    .value_counts(dropna=False)
    .rename_axis('monthly_income')
    .to_frame('n_clients')
)

print('Клиенты с отрицательным доходом:')
display(
    df_clients.loc[
        negative_income,
        ['client_id', 'monthly_income', 'income_source',
         'segment', 'loyalty_tier', 'city']
    ].sort_values('monthly_income')
)

Частоты отрицательных значений:


,n_clients
monthly_income,
-999.0,60


Клиенты с отрицательным доходом:


,client_id,monthly_income,income_source,segment,loyalty_tier,city
21,C001910,-999.0,Зарплата в нашем банке,Зарплатный,Золото,Санкт-Петербург
33,C000312,-999.0,Зарплата в нашем банке,Массовый,Серебро,Екатеринбург
611,C000346,-999.0,Зарплата в другом банке,Массовый,Базовый,Екатеринбург
661,C005147,-999.0,Зарплата в другом банке,Private,Золото,Нижний Новгород
670,C003626,-999.0,Зарплата в нашем банке,Массовый,Базовый,Москва
706,C003856,-999.0,Зарплата в другом банке,Массовый,Золото,Казань
857,C003465,-999.0,Зарплата в нашем банке,Зарплатный,Базовый,Москва
904,C005932,-999.0,Зарплата в нашем банке,Зарплатный,Базовый,Новосибирск
994,C002981,-999.0,Зарплата в другом банке,Массовый,Базовый,Нижний Новгород
1032,C004084,-999.0,Зарплата в нашем банке,Массовый,Платина,Казань


In [305]:
# Сохраняем исходные значения для проверки и журнала очистки
if 'monthly_income_raw' not in df_clients.columns:
    df_clients['monthly_income_raw'] = df_clients['monthly_income'].copy()

# Отмечаем клиентов, у которых исходный доход равен -999
df_clients['income_invalid'] = df_clients['monthly_income_raw'].eq(-999)

# Заменяем -999 на пропуск; остальные значения не меняем
df_clients['monthly_income'] = (
    df_clients['monthly_income'].replace(-999, np.nan)
)

print('Значений -999 в исходных данных:',
      df_clients['income_invalid'].sum())

print('Пропусков после обработки:',
      df_clients['monthly_income'].isna().sum())

print('Доля пропусков после обработки:',
      f"{df_clients['monthly_income'].isna().mean():.2%}")

# Пересчитываем статистику уже без -999
display(df_clients['monthly_income'].describe())

Значений -999 в исходных данных: 60
Пропусков после обработки: 612
Доля пропусков после обработки: 10.87%


count    5.020000e+03
mean     9.521502e+04
std      8.346974e+04
min      2.390000e+01
25%      5.187500e+04
50%      7.250000e+04
75%      1.074250e+05
max      1.120600e+06
Name: monthly_income, dtype: float64

Вывод: обнаружено 60 значений -999, можно предположить, что это техническое отсутствие сведений, поэтому заменили на NaN, чтобы не смещать распределение. Сейчас количество пропусков 10.87% выборки

## 1.7 Исследование отсутствующих сведений

In [306]:
# Приведём к единому виду income_source
source_text = (
    df_clients['income_source']
    .astype('string')
    .str.strip()  
    .str.casefold()  # нижний регистр
)

# отсутствует источник дохода 
df_clients['income_source_missing'] = (
    source_text.isna() | source_text.isin(['', 'не указан'])
)
# отсутствует оценка
df_clients['score_missing'] = df_clients['satisfaction_score'].isna()
# отсутствует ежемесячный доход
df_clients['income_missing'] = df_clients['monthly_income'].isna()

# Доли пропусков
display(
    df_clients[
        ['income_source_missing', 'score_missing', 'income_missing']
    ].agg(['sum', 'mean'])
)

,income_source_missing,score_missing,income_missing
sum,1536.000000,3376.000000,612.000000
mean,0.272727,0.599432,0.108665


Вывод: из 5632 клиентов источник дохода не указан у 1536 — 27,27%.
Оценка удовлетворённости отсутствует у 3376 — 59,94%.
Значение дохода отсутствует у 552 — 9,80%.